# 01. Daily REST API Extraction Task

Extracts previous day's order records (`created_at >= yesterday AND created_at < today`) in JSON format.

### Features:
- Pulls from configurable external REST API via `requests` with exponential retries.
- Automatic fallback to realistic mock generator if no external API is supplied.
- Partitions raw JSON directly into Unity Catalog landing volume:
  `/Volumes/bronze_prod/ecommerce/raw_landing/api_orders/year=YYYY/month=MM/day=DD/orders_YYYY-MM-DD.json`


In [0]:
# Load centralized configuration
%run ./01_Config


In [0]:
# Task Widgets
dbutils.widgets.text("p_date", "yesterday", "Extraction Date (YYYY-MM-DD or yesterday)")
dbutils.widgets.text("p_api_url", "", "API Base URL (Leave blank for mock)")
dbutils.widgets.text("p_api_key", "", "API Auth Key / Token")

p_date = dbutils.widgets.get("p_date")
p_api_url = dbutils.widgets.get("p_api_url")
p_api_key = dbutils.widgets.get("p_api_key")

print(f"[EXTRACT TASK] Target Date: '{p_date}', API URL: '{p_api_url or 'Built-in Mock'}'")


In [0]:
import json
import os
import random
from datetime import datetime, timedelta, timezone
from pathlib import Path
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def calculate_window(target_str):
    if target_str.lower() == "yesterday":
        now_utc = datetime.now(timezone.utc)
        target_date = (now_utc - timedelta(days=1)).date()
    else:
        target_date = datetime.strptime(target_str, "%Y-%m-%d").date()
    
    start_dt = datetime(target_date.year, target_date.month, target_date.day, 0, 0, 0, tzinfo=timezone.utc)
    end_dt = start_dt + timedelta(days=1)
    return start_dt, end_dt, target_date.strftime("%Y-%m-%d")

def generate_mock_payload(start_dt, end_dt, count=250):
    random.seed(int(start_dt.timestamp()))
    records = []
    statuses = ["COMPLETED", "COMPLETED", "COMPLETED", "SHIPPED", "PENDING", "CANCELLED", "RETURNED"]
    categories = ["Electronics", "Apparel", "Home & Kitchen", "Books", "Sports", "Beauty"]
    date_str_compact = start_dt.strftime("%Y%m%d")
    total_seconds = int((end_dt - start_dt).total_seconds())
    
    for idx in range(1, count + 1):
        order_id = f"ORD-API-{date_str_compact}-{idx:04d}"
        cust_id = f"C{random.randint(10001, 15000)}"
        prod_id = f"P{random.randint(1001, 1200)}"
        reg_id = random.randint(1, 8)
        qty = random.choices([1, 2, 3, 4, 5], weights=[0.55, 0.25, 0.12, 0.05, 0.03])[0]
        unit_price = round(random.uniform(9.99, 499.99), 2)
        total_amt = round(qty * unit_price, 2)
        status = random.choice(statuses)
        offset = random.randint(0, max(0, total_seconds - 1))
        created_at_dt = start_dt + timedelta(seconds=offset)
        created_at_str = created_at_dt.strftime("%Y-%m-%d %H:%M:%S")
        
        records.append({
            "order_id": order_id,
            "customer_id": cust_id,
            "product_id": prod_id,
            "region_id": reg_id,
            "quantity": qty,
            "unit_price": unit_price,
            "total_amount": total_amt,
            "order_status": status,
            "order_date": created_at_str,
            "created_at": created_at_str,
            "source_system": "REST_API",
            "category": random.choice(categories)
        })
    return records

start_dt, end_dt, date_str = calculate_window(p_date)
print(f"Extraction Window: {start_dt.isoformat()} <= created_at < {end_dt.isoformat()}")

if p_api_url and p_api_url.lower() != "mock":
    print(f"Fetching from external REST API: {p_api_url}")
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=Retry(total=3, backoff_factor=1)))
    headers = {"Accept": "application/json"}
    if p_api_key:
        headers["Authorization"] = f"Bearer {p_api_key}"
    resp = session.get(p_api_url, headers=headers, params={"created_at_gte": start_dt.isoformat(), "created_at_lt": end_dt.isoformat()}, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    records = data if isinstance(data, list) else data.get("data", [])
else:
    print("Using built-in mock order generator...")
    records = generate_mock_payload(start_dt, end_dt, count=300)

print(f"Total records retrieved: {len(records)}")


In [0]:
# Write JSON to Unity Catalog Volume
landing_api_dir = f"{RAW_LANDING_VOLUME}/api_orders/year={start_dt.year:04d}/month={start_dt.month:02d}/day={start_dt.day:02d}"
file_name = f"orders_{date_str}.json"
full_path = f"{landing_api_dir}/{file_name}"

# Ensure directory exists via dbutils
try:
    dbutils.fs.mkdirs(landing_api_dir)
except Exception as e:
    os.makedirs(landing_api_dir, exist_ok=True)

# Write newline-delimited JSON (NDJSON)
ndjson_content = "\n".join(json.dumps(r) for r in records) + "\n"

try:
    dbutils.fs.put(full_path, ndjson_content, overwrite=True)
    print(f"✓ Successfully staged {len(records)} JSON records to Volume via dbutils:")
    print(f"  {full_path}")
except Exception as e:
    # Fallback to local POSIX write (supported on UC single-node volumes)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, "w", encoding="utf-8") as f:
        f.write(ndjson_content)
    print(f"✓ Successfully wrote {len(records)} JSON records via OS filesystem:")
    print(f"  {full_path}")

dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "date": date_str, "records": len(records), "path": full_path}))
